In [3]:
import matplotlib
matplotlib.use('Agg')

import xarray as xr
import numpy as np

import matplotlib.pyplot as plt
import matplotlib.animation as animation
import cartopy.feature as cfeature
import cartopy.crs as ccrs

fil = "/Users/brianpm/Downloads/CERES_FLASH_TISA_Version1A_Subset_20260101-20260404.nc"

ds = xr.open_dataset(fil)

ds

<xarray.Dataset> Size: 1GB
Dimensions:                       (time: 94, lat: 180, lon: 360)
Coordinates:
  * time                          (time) datetime64[ns] 752B 2026-01-01 ... 2...
  * lat                           (lat) float32 720B -89.5 -88.5 ... 88.5 89.5
  * lon                           (lon) float32 1kB 0.5 1.5 2.5 ... 358.5 359.5
Data variables: (12/50)
    toa_sw_clr_daily              (time, lat, lon) float32 24MB ...
    toa_lw_clr_daily              (time, lat, lon) float32 24MB ...
    toa_net_clr_daily             (time, lat, lon) float32 24MB ...
    toa_alb_clr_daily             (time, lat, lon) float32 24MB ...
    toa_solar_clr_daily           (time, lat, lon) float32 24MB ...
    toa_sw_all_daily              (time, lat, lon) float32 24MB ...
    ...                            ...
    cldtemp_high_daily            (time, lat, lon) float32 24MB ...
    cldhght_high_daily            (time, lat, lon) float32 24MB ...
    cldpress_top_high_daily       (time, lat, lon) float32 24MB ...
    cldpress_base_high_daily      (time, lat, lon) float32 24MB ...
    lwp_high_daily                (time, lat, lon) float32 24MB ...
    iwp_high_daily                (time, lat, lon) float32 24MB ...
Attributes:
    title:        CERES FLASHFlux TISA Daily data.
    institution:  NASA Langley Research Center
    Conventions:  CF-1.4
    comment:      See values in latitude and longitudes dimensions.
    Version:      NOAA-20 Version 1A
    Fill_Value:   Fill Value is -999.0

In [4]:
ssta_fil = "/Users/brianpm/Downloads/sst.day.anom.2026.nc"
ds_ssta = xr.open_dataset(ssta_fil)

In [5]:

ndays = min(len(ds.time), len(ds_ssta.time))

clon, clat = np.meshgrid(ds.lon, ds.lat)
alon, alat = np.meshgrid(ds_ssta.lon, ds_ssta.lat)

fig, ax = plt.subplots(subplot_kw={'projection': ccrs.EckertIV()}, dpi=100)

cll0 = ds['cldarea_low_daily'].isel(time=0)
s0 = ds_ssta['anom'].isel(time=0)

img = ax.pcolormesh(clon, clat, cll0, transform=ccrs.PlateCarree(),
                    vmin=0, vmax=100, cmap='Grays_r')
contour_set = [ax.contour(alon, alat, s0, levels=[-4, -2, -1, 1, 2, 4],
                           transform=ccrs.PlateCarree(), cmap='RdBu_r', linewidths=0.75)]
ax.add_feature(cfeature.LAND, color='wheat', zorder=10)
fig.colorbar(img, ax=ax, shrink=0.4, label='Low Cloud Cover')
title = ax.set_title(str(ds.time.values[0])[:10])

out_path = "cloud_sstanom.mp4"
writer = animation.FFMpegWriter(fps=8, codec='h264', extra_args=['-crf', '28', '-preset', 'slow'])

with writer.saving(fig, out_path, dpi=100):
    for n in range(ndays):
        cll = ds['cldarea_low_daily'].isel(time=n).values
        s = ds_ssta['anom'].isel(time=n)

        img.set_array(cll.ravel())

        contour_set[0].remove()
        contour_set[0] = ax.contour(alon, alat, s, levels=[-4, -2, -1, 1, 2, 4],
                                     transform=ccrs.PlateCarree(), cmap='RdBu_r', linewidths=0.75)

        title.set_text(str(ds.time.values[n])[:10])
        writer.grab_frame()
        if n % 10 == 0:
            print(f"Frame {n}/{ndays}")

plt.close(fig)
print(f"Saved to {out_path}")

Frame 0/94
Frame 10/94
Frame 20/94
Frame 30/94
Frame 40/94
Frame 50/94
Frame 60/94
Frame 70/94
Frame 80/94
Frame 90/94
Saved to cloud_sstanom.mp4


In [6]:
# Regional animation: Hawaii to California, equator to 40N
# LambertConformal works well for this mid-latitude Pacific sector
extent = [-165, -105, 0, 42]
proj = ccrs.LambertConformal(central_longitude=-135, central_latitude=20,
                              standard_parallels=(10, 35))

fig, ax = plt.subplots(subplot_kw={'projection': proj}, figsize=(9, 5), dpi=100)
ax.set_extent(extent, crs=ccrs.PlateCarree())

cll0 = ds['cldarea_low_daily'].isel(time=0)
s0 = ds_ssta['anom'].isel(time=0)

img = ax.pcolormesh(clon, clat, cll0, transform=ccrs.PlateCarree(),
                    vmin=0, vmax=100, cmap='Grays_r')
contour_set = [ax.contour(alon, alat, s0, levels=[-4, -2, -1, 1, 2, 4],
                           transform=ccrs.PlateCarree(), cmap='RdBu_r', linewidths=0.75)]
ax.add_feature(cfeature.LAND, color='wheat', zorder=10)
ax.add_feature(cfeature.COASTLINE, linewidth=0.5, zorder=11)
ax.gridlines(draw_labels=True, linewidth=0.3, color='gray', alpha=0.5)
fig.colorbar(img, ax=ax, shrink=0.6, label='Low Cloud Cover (%)')
title = ax.set_title(str(ds.time.values[0])[:10])

out_path = "cloud_sstanom_pacific.mp4"
writer = animation.FFMpegWriter(fps=8, codec='h264', extra_args=['-crf', '28', '-preset', 'slow'])

with writer.saving(fig, out_path, dpi=100):
    for n in range(ndays):
        cll = ds['cldarea_low_daily'].isel(time=n).values
        s = ds_ssta['anom'].isel(time=n)

        img.set_array(cll.ravel())

        contour_set[0].remove()
        contour_set[0] = ax.contour(alon, alat, s, levels=[-4, -2, -1, 1, 2, 4],
                                     transform=ccrs.PlateCarree(), cmap='RdBu_r', linewidths=0.75)

        title.set_text(str(ds.time.values[n])[:10])
        writer.grab_frame()
        if n % 10 == 0:
            print(f"Frame {n}/{ndays}")

plt.close(fig)
print(f"Saved to {out_path}")

/Users/brianpm/miniforge3/envs/p12/lib/python3.14/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_land.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)
/Users/brianpm/miniforge3/envs/p12/lib/python3.14/site-packages/cartopy/io/__init__.py:242: DownloadWarning: Downloading: https://naturalearth.s3.amazonaws.com/50m_physical/ne_50m_coastline.zip
  warnings.warn(f'Downloading: {url}', DownloadWarning)


Frame 0/94
Frame 10/94
Frame 20/94
Frame 30/94
Frame 40/94
Frame 50/94
Frame 60/94
Frame 70/94
Frame 80/94
Frame 90/94
Saved to cloud_sstanom_pacific.mp4


In [10]:
ds_ceres = xr.open_dataset("/Users/brianpm/Library/CloudStorage/Dropbox/Data/CERES/Ed4.2/CERES_EBAF_Ed4.2_Subset_200003-201412.nc")
ds_ceres.data_vars

Data variables:
    toa_sw_all_mon               (time, lat, lon) float32 46MB ...
    toa_lw_all_mon               (time, lat, lon) float32 46MB ...
    toa_net_all_mon              (time, lat, lon) float32 46MB ...
    toa_sw_clr_c_mon             (time, lat, lon) float32 46MB ...
    toa_lw_clr_c_mon             (time, lat, lon) float32 46MB ...
    toa_net_clr_c_mon            (time, lat, lon) float32 46MB ...
    toa_sw_clr_t_mon             (time, lat, lon) float32 46MB ...
    toa_lw_clr_t_mon             (time, lat, lon) float32 46MB ...
    toa_net_clr_t_mon            (time, lat, lon) float32 46MB ...
    toa_cre_sw_mon               (time, lat, lon) float32 46MB ...
    toa_cre_lw_mon               (time, lat, lon) float32 46MB ...
    toa_cre_net_mon              (time, lat, lon) float32 46MB ...
    solar_mon                    (time, lat, lon) float32 46MB ...
    cldarea_total_daynight_mon   (time, lat, lon) float32 46MB ...
    cldpress_total_daynight_mon  (time, lat, l

In [ ]:
ds_ceres['low']